# Data Leakage 
Leakage = the model sees information at training/evaluation time that would
not exist at prediction time, so scores look great and reality is poor.

Sections (following the doc):
1. Tools: logistic regression, StandardScaler, split, Pipeline
2. Target leakage (house price / exam score)
3. Preprocessing leakage (scale before split; feature selection before split)
4. Train-test contamination (duplicates)
5. Temporal leakage (random split vs time split)
6. Prevention: split first, pipeline, feature audit
7. Signs of leakage and leakage vs overfitting

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(42)

## 1. Tools

In [ ]:
class StandardScaler:
    """z = (x - mean) / std, statistics learned by fit() only."""
    def fit(self, X):
        self.mean_ = X.mean(0); self.std_ = X.std(0) + 1e-12; return self
    def transform(self, X): return (X - self.mean_) / self.std_
    def fit_transform(self, X): return self.fit(X).transform(X)

class LogisticRegression:
    """Binary logistic regression trained by batch gradient descent."""
    def __init__(self, lr=0.1, epochs=500, l2=0.0):
        self.lr, self.epochs, self.l2 = lr, epochs, l2
    @staticmethod
    def _sig(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    def fit(self, X, y):
        n, d = X.shape; self.w = np.zeros(d); self.b = 0.0
        for _ in range(self.epochs):
            g = self._sig(X @ self.w + self.b) - y
            self.w -= self.lr * (X.T @ g / n + self.l2 * self.w)
            self.b -= self.lr * g.mean()
        return self
    def predict_proba(self, X): return self._sig(X @ self.w + self.b)
    def predict(self, X): return (self.predict_proba(X) >= .5).astype(int)
    def score(self, X, y): return np.mean(self.predict(X) == y)

class LinearRegression:
    def fit(self, X, y):
        A = np.column_stack([np.ones(len(X)), X])
        self.coef = np.linalg.lstsq(A, y, rcond=None)[0]; return self
    def predict(self, X): return np.column_stack([np.ones(len(X)), X]) @ self.coef

class Pipeline:
    """Chain of (scaler, model): fit() only ever touches the data it is given,
    so calling pipe.fit(X_train) cannot leak test statistics."""
    def __init__(self, scaler, model): self.scaler, self.model = scaler, model
    def fit(self, X, y):
        self.model.fit(self.scaler.fit_transform(X), y); return self
    def predict(self, X): return self.model.predict(self.scaler.transform(X))
    def score(self, X, y): return self.model.score(self.scaler.transform(X), y)

def train_test_split(X, y, test_size=.3, rng=rng, shuffle=True):
    idx = rng.permutation(len(X)) if shuffle else np.arange(len(X))
    cut = int(len(X) * (1 - test_size))
    return X[idx[:cut]], X[idx[cut:]], y[idx[:cut]], y[idx[cut:]]

def r2(y, p): return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

## 2. Target leakage
House price: including `final_price` (or something derived from it, like
`price_per_sqft`) as a feature makes the model look perfect. Exam pass/fail:
`final_exam_score` is only known after the outcome.

In [ ]:
n = 600
area = rng.uniform(50, 250, n); bed = rng.integers(1, 6, n); age = rng.uniform(0, 60, n)
price = 2000 * area + 10000 * bed - 800 * age + rng.normal(0, 40000, n)
X_ok = np.column_stack([area, bed, age])
X_leak = np.column_stack([X_ok, price / area + rng.normal(0, 1, n)])  # price/sqft leaks

def eval_lr(X):
    Xtr, Xte, ytr, yte = train_test_split(X, price)
    sc = StandardScaler().fit(Xtr)
    m = LinearRegression().fit(sc.transform(Xtr), ytr)
    return r2(yte, m.predict(sc.transform(Xte)))
r2_ok, r2_leak = eval_lr(X_ok), eval_lr(X_leak)
print(f"House price  R2 honest={r2_ok:.3f}  with leaked feature={r2_leak:.3f}")

# Student pass/fail
study = rng.normal(5, 2, n); attend = rng.uniform(.4, 1, n); prev = rng.normal(65, 12, n)
latent = .5 * study + 4 * attend + .05 * prev + rng.normal(0, 1.2, n)
passed = (latent > np.median(latent)).astype(int)
final_exam = 40 + 30 * passed + rng.normal(0, 5, n)      # determined by the outcome
Xs_ok = np.column_stack([study, attend, prev]); Xs_leak = np.column_stack([Xs_ok, final_exam])
def eval_clf(X, y):
    Xtr, Xte, ytr, yte = train_test_split(X, y)
    return Pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr).score(Xte, yte)
acc_ok, acc_leak = eval_clf(Xs_ok, passed), eval_clf(Xs_leak, passed)
print(f"Student pass accuracy honest={acc_ok:.3f}  with final_exam leaked={acc_leak:.3f}")

## 3. Preprocessing leakage
(a) Scaling on all data before splitting: mean/std of the test set seep in.
The effect is usually small; we show the *difference* in the fitted stats.
(b) Feature selection on all data before splitting: dramatic. With PURE NOISE
features and random labels, true accuracy is 50%, but selecting the "best"
features using all labels first yields much higher test accuracy.

In [ ]:
# (a) scaler statistics
Xd = rng.normal(0, 1, (100, 3)) + np.array([0, 0, 0])
Xd[70:] += 2                                   # "test" rows come from a shifted region
sc_all = StandardScaler().fit(Xd); sc_tr = StandardScaler().fit(Xd[:70])
print("Scaler mean using ALL data  :", np.round(sc_all.mean_, 2))
print("Scaler mean using TRAIN only:", np.round(sc_tr.mean_, 2), "(what deployment would know)")

# (b) feature selection leakage on noise
def top_k_features(X, y, k):
    """Rank features by |correlation with y| (filter method)."""
    Xc = (X - X.mean(0)) / (X.std(0) + 1e-12); yc = (y - y.mean()) / y.std()
    return np.argsort(-np.abs(Xc.T @ yc / len(y)))[:k]

def selection_experiment(seed, n=100, d=2000, k=10):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, d)); y = r.integers(0, 2, n)   # labels independent of X
    half = n // 2
    Xtr, Xte, ytr, yte = X[:half], X[half:], y[:half], y[half:]
    # WRONG: select with all data, then split
    f_bad = top_k_features(X, y, k)
    a_bad = Pipeline(StandardScaler(), LogisticRegression(epochs=300)).fit(Xtr[:, f_bad], ytr).score(Xte[:, f_bad], yte)
    # RIGHT: select with training data only
    f_ok = top_k_features(Xtr, ytr, k)
    a_ok = Pipeline(StandardScaler(), LogisticRegression(epochs=300)).fit(Xtr[:, f_ok], ytr).score(Xte[:, f_ok], yte)
    return a_bad, a_ok
res = np.array([selection_experiment(s) for s in range(20)])
print(f"Noise data: leaky selection acc={res[:,0].mean():.3f}  correct={res[:,1].mean():.3f}  (truth = 0.5)")

plt.figure(figsize=(5, 4))
plt.bar(["leaky\nselection", "correct\n(train only)"], res.mean(0), color=["tab:red", "tab:green"])
plt.axhline(.5, color='k', ls='--', label="true chance"); plt.ylim(0, 1); plt.legend()
plt.title("Feature selection before split (random labels)")
plt.tight_layout(); plt.savefig('plots/01_selection_leakage.png', dpi=110); plt.close()

## 4. Train-test contamination (duplicates)
If duplicate rows straddle the split, a memorising model (1-NN) looks perfect
on 'test' data it has effectively seen.

In [ ]:
def one_nn_predict(Xtr, ytr, Xte):
    d = ((Xte[:, None] - Xtr[None]) ** 2).sum(2)
    return ytr[np.argmin(d, 1)]

Xn = rng.normal(size=(300, 5)); yn = (Xn[:, 0] + rng.normal(0, 1.5, 300) > 0).astype(int)  # noisy labels
# duplicate 40% of the rows BEFORE splitting
dup = rng.choice(300, 120, replace=False)
Xd2 = np.vstack([Xn, Xn[dup]]); yd2 = np.concatenate([yn, yn[dup]])
Xtr, Xte, ytr, yte = train_test_split(Xd2, yd2)
acc_dup = np.mean(one_nn_predict(Xtr, ytr, Xte) == yte)
# clean: split first, then no duplicates across
Xtr, Xte, ytr, yte = train_test_split(Xn, yn)
acc_clean = np.mean(one_nn_predict(Xtr, ytr, Xte) == yte)
print(f"1-NN accuracy with duplicates across split={acc_dup:.3f}, clean split={acc_clean:.3f}")

## 5. Temporal leakage
Predict tomorrow's value of a trending, autocorrelated series. A random split
lets the model interpolate between future and past neighbours (leak); a
chronological split is the honest estimate.

In [ ]:
T = 400
t = np.arange(T)
series = np.cumsum(rng.normal(0, 1, T)) + 0.05 * t
# features: value at t-1 and t+1 (t+1 is FUTURE info!), target: value at t
tt = t[1:-1]; y_t = series[1:-1]
X_past = series[:-2, None]
X_fut = np.column_stack([series[:-2], series[2:]])       # includes future value
def fit_eval(X, split):
    if split == 'random':
        Xtr, Xte, ytr, yte = train_test_split(X, y_t)
    else:
        c = int(.7 * len(y_t)); Xtr, Xte, ytr, yte = X[:c], X[c:], y_t[:c], y_t[c:]
    m = LinearRegression().fit(Xtr, ytr)
    return np.sqrt(np.mean((yte - m.predict(Xte)) ** 2))
rows = {"past only + time split (honest)": fit_eval(X_past, 'time'),
        "past only + random split": fit_eval(X_past, 'random'),
        "uses FUTURE feature + random split": fit_eval(X_fut, 'random')}
for k, v in rows.items(): print(f"RMSE {v:6.3f}  {k}")

plt.figure(figsize=(9, 3.5))
plt.plot(t, series, lw=1); plt.axvline(.7 * T, color='r', ls='--', label='chronological split')
plt.legend(); plt.title("Time series: never train on data after the test period")
plt.tight_layout(); plt.savefig('plots/02_temporal.png', dpi=110); plt.close()

## 6. Prevention: split first, fit preprocessing on train only, audit features
A helper that flags suspicious features: a single feature that alone predicts
the target near-perfectly (very high |correlation|) deserves a "would this be
known at prediction time?" question.

In [ ]:
def audit_features(X, y, names, thresh=0.9):
    """Return features whose |corr(x, y)| exceeds thresh (leakage suspects)."""
    out = []
    for j, nme in enumerate(names):
        c = abs(np.corrcoef(X[:, j], y)[0, 1])
        out.append((nme, c, c > thresh))
    return out
for nme, c, flag in audit_features(Xs_leak, passed, ["study_hours", "attendance", "prev_grade", "final_exam"], 0.7):
    print(f"{nme:12s} |corr|={c:.2f} {'<-- SUSPECT: known only after the outcome?' if flag else ''}")

# Correct workflow using Pipeline
Xtr, Xte, ytr, yte = train_test_split(Xs_ok, passed)
pipe = Pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
print("Correct workflow test accuracy:", round(pipe.score(Xte, yte), 3))

## 7. Signs of leakage; leakage vs overfitting
* Leakage: HIGH test score, collapses in production.
* Overfitting: LOW train error, HIGH test error.
We simulate 'deployment' data (leaked feature unavailable -> set to its mean).

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xs_leak, passed)
leaky = Pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
X_deploy = Xte.copy(); X_deploy[:, 3] = Xtr[:, 3].mean()   # exam not yet taken
acc_lab, acc_prod = leaky.score(Xte, yte), leaky.score(X_deploy, yte)
print(f"Leaky model: test acc={acc_lab:.3f}  -> production acc={acc_prod:.3f}")

# overfitting contrast: 1-NN on noisy data, train acc 1.0 but test acc low
Xo = rng.normal(size=(200, 5)); yo = (Xo[:, 0] + rng.normal(0, 2, 200) > 0).astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xo, yo)
print(f"Overfit 1-NN: train acc={np.mean(one_nn_predict(Xtr, ytr, Xtr)==ytr):.3f} test acc={np.mean(one_nn_predict(Xtr, ytr, Xte)==yte):.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
vals = [acc_ok, acc_lab, acc_prod]
ax.bar(["honest\nfeatures", "leaky\n(test)", "leaky model\nin production"], vals, color=["tab:green", "tab:red", "tab:gray"])
ax.set_ylim(0, 1.05); ax.set_ylabel("accuracy"); ax.set_title("Leakage: great in the lab, poor in reality")
plt.tight_layout(); plt.savefig('plots/03_leaky_vs_real.png', dpi=110); plt.close()

# Optional sklearn sanity check
try:
    from sklearn.linear_model import LogisticRegression as SkLR
    Xtr, Xte, ytr, yte = train_test_split(Xs_ok, passed)
    sc = StandardScaler().fit(Xtr)
    print("sklearn LR acc:", round(SkLR().fit(sc.transform(Xtr), ytr).score(sc.transform(Xte), yte), 3),
          "| ours:", round(Pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr).score(Xte, yte), 3))
except Exception as e:
    print("sklearn check skipped:", e)